In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("powerplant_data.csv")

In [ ]:
df.head(5)

In [ ]:
# AT => temperature
# V => vacuum
# AP => pressure
# RH => humidity

# PE => produced energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop(columns = "PE")
y = df["PE"]

In [ ]:
X.head(2)

In [ ]:
y.head(2)

In [ ]:
#split 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [ ]:
X_test

In [ ]:
df.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

In [ ]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1, 1) # -1 is the row size which auto adapts to
                                                                                 # our rows and 1 is the column size

X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1, 1)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

test_loader = DataLoader(test_dataset, batch_size = 32)

# DEEP LEARNING

In [ ]:
#Define our ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
    
            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),
    
            # output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch.optim as optim

In [ ]:
model = ANN()

# loss, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train The ANN
train_losses = []
val_losses = []
best_val_loss = float("inf")

epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # Total running loss for 1 epoch

    for xb, yb in train_loader:
        # xb = features of 1 branch
        # yb = labels of 1 branch

        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop... predicted output for this batch
        loss = critetrion(outputs, yb) # Compute loss
        loss.backward() # Backward prop... compute gradients
        optimizer.step() # Update params

        running_loss += loss.item() #Loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():  # Number of gradient compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> Train Loss = {epoch_train_loss} & Val Loss {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
       # torch.save(model.state_dict(), "best_model.pt")  #.pt or .pth   
        

In [ ]:
# Plotting losses

import matplotlib.pyplot as plt
loss_df = pd.DataFrame({
    "Training Loss": train_losses, 
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validatiion Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
# Loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
# Evaluation
model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = crietrion(train_preds, y_train_tensor)
    test_mse_loss = crietrion(test_preds, y_test_tensor)

print("Training MSE: ", train_mse_loss.item())
print("Testing MSE: ", test_mse_loss.item())